# Bing Grounding Tool

Creates a persistent Foundry agent with the **Bing Grounding** tool enabled.
When the user asks about recent events the agent searches Bing automatically.

In [ ]:
%pip install azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import AgentStreamEvent, BingGroundingTool
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

endpoint = os.environ["AZURE_FOUNDRY_ENDPOINT"]
bing_connection_id = os.environ["BING_CONNECTION_ID"]

client = AIProjectClient(endpoint=endpoint, credential=DefaultAzureCredential())
print("Client ready.")

In [ ]:
# Define the Bing Grounding tool and create an agent with it
bing_tool = BingGroundingTool(connection_id=bing_connection_id)

agent = client.agents.create_agent(
    model="gpt-4.1",
    name="GroundedSearchAgent",
    instructions="You answer questions using Bing search to find current, accurate information.",
    tools=bing_tool.definitions,
)
print(f"Agent created: {agent.id}")

In [ ]:
# Ask a question that requires live information
thread = client.agents.create_thread()
client.agents.create_message(
    thread_id=thread.id,
    role="user",
    content="What are the latest major developments in AI in the past month?",
)

# Stream the agent's response
print("--- Agent Response ---\n")

with client.agents.create_stream(thread_id=thread.id, agent_id=agent.id) as stream:
    for event_type, event_data, _ in stream:
        if event_type == AgentStreamEvent.THREAD_MESSAGE_DELTA:
            for block in event_data.delta.content:
                if block.type == "text":
                    print(block.text.value, end="", flush=True)

In [ ]:
# Cleanup: delete the agent when done
client.agents.delete_agent(agent_id=agent.id)
print("\nAgent deleted.")